# Inspect workflow evidence

Cache a policy activation and inspect the recorded execution provenance.

In [ ]:
import torch
from tensordict import TensorDict
from tensordict.nn import TensorDictModule
from tdhook.latent import ActivationCaching
from tdhook.workflow import Workflow
from xdrl import (
    BatchSemantics,
    InteractionContract,
    InteractionPhase,
    KeyPresence,
    KeyRole,
    KeySchema,
    ModelRole,
    RuntimeInteractionContext,
    TDHookWorkflowRunner,
    TensorDictSchema,
    WorkflowProvenance,
)

torch.manual_seed(0)
batch = TensorDict({"observation": torch.randn(4, 4)}, batch_size=[4])
policy = TensorDictModule(
    torch.nn.Linear(4, 2),
    in_keys=["observation"],
    out_keys=["action"],
)

In [ ]:
batch_dims = BatchSemantics(("env",))
contract = InteractionContract(
    identity="policy:evaluation",
    role=ModelRole.ACTOR,
    phase=InteractionPhase.EVALUATION,
    module_path="policy",
    input_schema=TensorDictSchema(
        (KeySchema("observation", KeyRole.OBSERVATION, KeyPresence.REQUIRED),),
        batch_dims,
    ),
    output_schema=TensorDictSchema(
        (KeySchema("action", KeyRole.ACTION, KeyPresence.PRODUCED),),
        batch_dims,
    ),
    module_training=False,
)
interaction = RuntimeInteractionContext(contract, policy, batch)

In [ ]:
workflow = Workflow(ActivationCaching("module", cache_key=("activations", "head")))
execution = TDHookWorkflowRunner(interaction).run(
    workflow,
    batch.clone(),
    code_revision="tutorial",
)
activation = execution.data["activations", "head", "module"]

assert activation.shape == (4, 2)
assert execution.provenance.model_calls == 1
assert WorkflowProvenance.from_json(execution.provenance.to_json()) == execution.provenance
{
    "action_shape": tuple(execution.data["action"].shape),
    "activation_shape": tuple(activation.shape),
    "model_calls": execution.provenance.model_calls,
    "lifecycle": [event.kind.value for event in interaction.events],
}